# Computer-Using Agents

## Scenario: safely draft and submit a Northstar support escalation

Computer-use agents bridge the gap between a model and existing graphical software. They are powerful precisely because they operate ordinary interfaces; they are risky for the same reason. This notebook uses a deterministic sandbox simulation—**never a live browser or OS**—to teach visual grounding, mouse/keyboard-like actions, confirmation, and recovery.

**Outcomes:** distinguish DOM automation from visual computer use; ground actions on a fresh UI state; enforce sandbox, policy, and confirmation boundaries; recover safely from UI drift; and measure trajectory quality.

## The control loop

![Safe computer-use agent control loop](assets/computer-use-control-loop.svg)

A model may suggest an action, but the application is the authority. It captures a fresh observation, resolves intent to one visible target, checks origin/risk/budgets, executes only in an isolated environment, verifies the postcondition, and stops or recovers. The diagram is a static SVG so it renders in GitHub notebooks without a Mermaid extension.

## 1. Choose the interaction surface

Start with a typed API. It is normally more reliable, cheaper, and easier to authorize than UI control. Choose DOM/accessibility automation for owned pages with stable semantic contracts; use visual computer interaction only where an interface is genuinely UI-only or visual state matters. Desktop and mobile agents expand capability across applications but also expand the permission, secret, file, and egress boundary.

A native computer-use model can understand screenshots and propose clicks/types/scrolls. It cannot safely self-authorize a form submission, a login, a purchase, or a cross-origin navigation.

In [ ]:
from lab import Action, SandboxPolicy, Session, apply_action, dom_click, run_safe_support_flow, screenshot_summary

session = Session(ui_variant='renamed')
for item in screenshot_summary(session):
    print(item)

## 2. Visual grounding is not a raw coordinate

A safe click binds a task to the **current** visible element: semantic role, accessible/visible label, nearby context, element bounds, origin, and an optional coordinate verified inside those bounds. The controller rejects zero or many matches. This protects against stale screens, overlays, and a model clicking an arbitrary region.

The support portal has changed the button label from `Escalate` to `Create escalation draft`. A brittle selector fails rather than silently doing something else.

In [ ]:
session.screen = 'case'
try:
    dom_click(session, '#escalate-button')
except LookupError as error:
    print('Controlled selector failure:', error)

# Semantic grounding survives the label change, but still requires a unique visible target.
policy = SandboxPolicy()
result = apply_action(session, policy, Action('click', target='draft', expected_label='escalation'))
print(result, '→ current screen:', session.screen)

## 3. Mouse, keyboard, and navigation need typed contracts

Do not give the model a shell-like `browser(command)` tool. Use narrow action objects: `navigate` to an allowlisted route, `click` on a grounded target, `type` into a specific field, and `confirm` for a pending commit. The policy in `lab.py` verifies origin, action allowlist, action budget, target uniqueness, and risk tier before applying a state transition.

A browser controller also needs fresh screenshots after navigation, scroll, a modal, or timeout. A production OS agent adds process, clipboard, filesystem, download/upload, window, network-egress, and secret scopes. A mobile controller adds device/app/package/deep-link boundaries and must handle smaller, more ambiguous targets.

In [ ]:
# An out-of-bounds coordinate is rejected even if the label is otherwise correct.
unsafe = Session(screen='case', ui_variant='renamed')
try:
    apply_action(unsafe, SandboxPolicy(), Action('click', target='draft', expected_label='escalation', coordinates=(999, 999)))
except ValueError as error:
    print('Grounding protection:', error)

## 4. Sandboxing and confirmation

Read and draft actions can be low-risk only inside a narrow task scope. A commit action—submit, send, delete, purchase, access change—must pause with an action digest that identifies the origin, target, payload hash, actor/tenant, policy version, evidence snapshot, risk tier, and expiry. If any value changes, invalidate approval. Sensitive login/MFA/payment flows should normally hand control to the user.

The simulation keeps the submit action pending until a human applies `confirm`. The model never supplies its own confirmation identity.

In [ ]:
safe_run = run_safe_support_flow(ui_changed=True)
print('Submitted after human confirmation:', safe_run.submitted)
print('\nTrace:')
print('\n'.join(safe_run.actions))
assert safe_run.submitted
assert any('paused for confirmation' in event for event in safe_run.actions)

## 5. Recovering from UI changes

When a selector disappears, an element moves, an unexpected modal appears, or navigation changes origin: stop the stale action, obtain a new observation, compare expected and actual state, and perform at most a policy-approved recovery. If the new target is ambiguous, consequential, or outside the allowed origin, pause for a human. Retrying an unverified click is not recovery.

Treat webpage contents—including hidden text and prompts—as untrusted data. They cannot change a task, enable a tool, request credentials, or authorize navigation. Computer-use agents must also have step/time/cost budgets and a trace containing snapshots, proposed actions, policy decisions, confirmations, postconditions, and terminal reason.

## Evaluation and deployment checklist

Evaluate target grounding, state-transition/postcondition correctness, confirmation coverage for commit actions, recovery success after layout/label changes, blocked cross-origin attempts, duplicate-action prevention, action count, latency, and task success. Use isolated benchmark environments—not production users and accounts—for browser/desktop evaluation.

- Use an API when one exists.
- Use an ephemeral browser profile, container, or VM with scoped credentials and constrained egress.
- Require fresh grounding before each consequential action.
- Verify the postcondition after every navigation, form submission, and state change.
- Cap retries and require idempotency/postcondition checks before repeating a write.
- Require human confirmation or takeover for commits and sensitive input.

## Exercises

1. Add a modal that obscures the submit button. Make the controller reobserve and refuse to click through it.
2. Add an unallowlisted redirect and test that the policy blocks navigation before a screenshot is captured.
3. Add a mobile `tap` action with a smaller target; document the extra accessibility and confirmation requirements.
4. Design a DOM-first, vision-only, and hybrid controller for the same portal. Which has the best reliability/security trade-off?

## Sources

- [OpenAI computer-use guide](https://developers.openai.com/api/docs/guides/tools-computer-use)
- [OpenAI Operator/CUA safety and takeover design](https://openai.com/index/introducing-operator/)
- [Anthropic computer-use documentation](https://docs.anthropic.com/en/docs/build-with-claude/computer-use)
- [OSWorld benchmark](https://arxiv.org/abs/2404.07972)
- [WebArena benchmark](https://arxiv.org/abs/2307.13854)
- [BrowserGym](https://github.com/ServiceNow/BrowserGym)